In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 76.3 MB/s eta 0:00:00


In [ ]:

from google.colab import drive

drive.mount('/content/mydrive')

Mounted at /content/mydrive


In [ ]:
import os

data_path = "/content/mydrive/MyDrive/cryptocast/data/processed/bitcoin_cleaned.csv"

print(os.path.exists(data_path))

True


In [ ]:
import os

models_path = "/content/mydrive/MyDrive/cryptocast/streamlit/models"

print(os.listdir(models_path))

['transformer_1d.keras', 'transformer_3d.keras', 'transformer_7d.keras', 'scaler_1d.pkl', 'scaler_3d.pkl', 'scaler_7d.pkl', 'cnn_3d.keras', 'cnn_7d.keras', 'cnn_scaler_3d.pkl', 'cnn_scaler_7d.pkl', 'rnn_1d.keras', 'rnn_3d.keras', 'rnn_7d.keras', 'rnn_scaler_1d.pkl', 'rnn_scaler_3d.pkl', 'rnn_scaler_7d.pkl', 'lstm_1d.keras', 'lstm_3d.keras', 'lstm_7d.keras', 'lstm_scaler_1d.pkl', 'lstm_scaler_3d.pkl', 'lstm_scaler_7d.pkl', 'cnn_1d.keras', 'cnn_scaler.pkl']


In [ ]:
!pip install -q streamlit pyngrok tensorflow scikit-learn pandas numpy

In [ ]:
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

TensorFlow: 2.20.0
Keras: 3.13.2


In [ ]:
from pathlib import Path

APP_DIR = Path("/content/mydrive/MyDrive/cryptocast/streamlit")
MODEL_DIR = APP_DIR / "models"
ASSETS_DIR = APP_DIR / "assets"
DATA_PATH = Path(
    "/content/mydrive/MyDrive/cryptocast/data/processed/bitcoin_cleaned.csv"
)

print("App:", APP_DIR.exists())
print("Models:", MODEL_DIR.exists())
print("Assets:", ASSETS_DIR.exists())
print("Dataset:", DATA_PATH.exists())

App: True
Models: True
Assets: True
Dataset: True


In [ ]:
%%writefile /content/mydrive/MyDrive/cryptocast/streamlit/app.py

# CryptoCast - Gold & Emerald UI
# Complete app.py
# Existing models and prediction logic preserved

import os
import base64
import joblib
import numpy as np
import pandas as pd
import streamlit as st
import tensorflow as tf

from pathlib import Path
from tensorflow.keras.models import load_model

# ============================================
# TRANSFORMER CUSTOM LAYER
# ============================================

@tf.keras.utils.register_keras_serializable(package="Custom")
class PositionalEncoding(tf.keras.layers.Layer):

    def __init__(self, sequence_length=60, d_model=64, **kwargs):
        super().__init__(**kwargs)
        self.sequence_length = sequence_length
        self.d_model = d_model

    def call(self, inputs):

        seq_len = tf.shape(inputs)[1]
        depth = tf.shape(inputs)[2]

        positions = tf.cast(
            tf.range(seq_len)[:, None],
            tf.float32
        )

        dimensions = tf.cast(
            tf.range(depth)[None, :],
            tf.float32
        )

        rates = tf.pow(
            10000.0,
            -(2.0 * tf.floor(dimensions / 2.0))
            / tf.cast(depth, tf.float32)
        )

        angles = positions * rates

        even = tf.equal(
            tf.math.floormod(
                tf.range(depth),
                2
            ),
            0
        )[None, :]

        encoding = tf.where(
            even,
            tf.sin(angles),
            tf.cos(angles)
        )

        return inputs + encoding[None, :, :]

    def get_config(self):

        config = super().get_config()

        config.update({
            "sequence_length": self.sequence_length,
            "d_model": self.d_model
        })

        return config

# ============================================
# CONFIGURATION
# ============================================

st.set_page_config(
    page_title="CryptoCast | Bitcoin Forecasting",
    page_icon="₿",
    layout="wide",
    initial_sidebar_state="expanded"
)



APP_DIR = "/content/mydrive/MyDrive/cryptocast/streamlit"

MODEL_DIR = os.path.join(APP_DIR, "models")
ASSETS_DIR = os.path.join(APP_DIR, "assets")

DATA_PATH = "/content/mydrive/MyDrive/cryptocast/data/processed/bitcoin_cleaned.csv"
print("App directory:", APP_DIR)
print("Models directory:", MODEL_DIR)
print("Assets directory:", ASSETS_DIR)

print("Models folder exists:", os.path.exists(MODEL_DIR))
print("Assets folder exists:", os.path.exists(ASSETS_DIR))
print("Dataset exists:", os.path.exists(DATA_PATH))
LOOKBACK = 60

FEATURES = [
    "Price",
    "Open",
    "High",
    "Low",
    "Vol_Log",
    "Change %"
]

MODEL_CONFIG = {
    "CNN": {
        1: ("cnn_1d.keras", "cnn_scaler.pkl"),
        3: ("cnn_3d.keras", "cnn_scaler_3d.pkl"),
        7: ("cnn_7d.keras", "cnn_scaler_7d.pkl")
    },
    "RNN": {
        1: ("rnn_1d.keras", "rnn_scaler_1d.pkl"),
        3: ("rnn_3d.keras", "rnn_scaler_3d.pkl"),
        7: ("rnn_7d.keras", "rnn_scaler_7d.pkl")
    },
    "LSTM": {
        1: ("lstm_1d.keras", "lstm_scaler_1d.pkl"),
        3: ("lstm_3d.keras", "lstm_scaler_3d.pkl"),
        7: ("lstm_7d.keras", "lstm_scaler_7d.pkl")
    },
    "Transformer": {
        1: ("transformer_1d.keras", "scaler_1d.pkl"),
        3: ("transformer_3d.keras", "scaler_3d.pkl"),
        7: ("transformer_7d.keras", "scaler_7d.pkl")
    }
}

# ============================================
# PREMIUM CRYPTOCAST UI
# ============================================

def get_banner_image():
    for ext in (".png", ".jpg", ".jpeg", ".webp"):
        image_path = os.path.join(
            ASSETS_DIR, "bitcoin_banner" + ext
        )

        if os.path.isfile(image_path):
            with open(image_path, "rb") as f:
                encoded = base64.b64encode(
                    f.read()
                ).decode("utf-8")

            mime = (
                "jpeg"
                if ext in (".jpg", ".jpeg")
                else ext.lstrip(".")
            )

            return f"data:image/{mime};base64,{encoded}"

    return None


banner = get_banner_image()

background_layer = (
    f'url("{banner}")'
    if banner
    else "linear-gradient(135deg,#071b15,#123c2b)"
)

st.markdown(f"""
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Manrope:wght@400;500;600;700;800&display=swap');

:root {{
    --gold: #dfbd65;
    --gold-light: #f6e7b0;
    --text: #f3f5e9;
    --muted: #c5d5c8;
    --border: rgba(223,189,101,.26);
}}


.stApp {{
    background:
        linear-gradient(
            110deg,
            rgba(3,18,13,.91),
            rgba(5,28,20,.84)
        ),
        {background_layer};
    background-size: cover;
    background-position: center;
    background-attachment: fixed;
    color: var(--text);
    font-family: 'DM Sans',sans-serif;
}}

[data-testid="stHeader"] {{
    background: rgba(4,20,14,.45);
}}

[data-testid="stSidebar"] {{
    background: rgba(5,25,18,.97);
    border-right: 1px solid var(--border);
}}

[data-testid="stSidebar"] * {{
    color: var(--text);
}}

.block-container {{
    max-width: 1450px;
    padding-top: 2rem;
    padding-bottom: 3rem;
}}

h1,h2,h3 {{
    font-family: 'Manrope',sans-serif !important;
    color: var(--gold-light) !important;
}}

p,label,.stCaption {{
    color: var(--muted);
}}

.hero {{
    padding: 30px 34px;
    border: 1px solid rgba(223,189,101,.34);
    border-radius: 22px;
    background:
        linear-gradient(
            120deg,
            rgba(8,39,27,.95),
            rgba(17,59,40,.79)
        );
    box-shadow: 0 12px 40px rgba(0,0,0,.25);
    margin-bottom: 26px;
}}

.hero-label,.section-label {{
    color: var(--gold);
    font-size: 12px;
    font-weight: 700;
    letter-spacing: 2.5px;
    text-transform: uppercase;
}}

.hero-title {{
    color: #fff4cb;
    font-family: 'Manrope',sans-serif;
    font-size: clamp(32px,5vw,48px);
    font-weight: 800;
    line-height: 1.2;
    margin: 12px 0;
}}

.hero-desc {{
    color: #d0e0d4;
    font-size: 15px;
    max-width: 760px;
    line-height: 1.7;
}}
.forecast-heading {{
    margin-top: 5px;
    margin-bottom: 6px;
    font-size: clamp(28px, 4vw, 40px);
    line-height: 1.15;
    color: #fff4cb !important;
    font-family: 'Manrope', sans-serif !important;
    font-weight: 800;
}}

.hero-badges {{
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
    margin-top: 22px;
}}

.hero-badge {{
    padding: 7px 12px;
    border-radius: 999px;
    border: 1px solid rgba(223,189,101,.20);
    background: rgba(3,24,17,.48);
    color: #d9e7df;
    font-size: 11px;
    font-weight: 600;
}}

.forecast-meta {{
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
    margin: 4px 0 18px;
}}

.forecast-chip {{
    padding: 7px 11px;
    border-radius: 999px;
    background: rgba(11,51,35,.78);
    border: 1px solid rgba(53,201,139,.20);
    color: #cfe8dc;
    font-size: 12px;
}}

.forecast-chip strong {{
    color: #f6e7b0;
}}

.sidebar-brand {{
    padding: 6px 4px 22px;
}}

.sidebar-brand-row {{
    display: flex;
    align-items: center;
    gap: 10px;
}}

.sidebar-dot {{
    width: 16px;
    height: 16px;
    border-radius: 50%;
    background: #f6df17;
}}

.sidebar-brand-title {{
    font-family: 'Manrope', sans-serif;
    font-size: 23px;
    font-weight: 800;
    color: #fff9df;
}}

.sidebar-subtitle {{
    margin-top: 8px;
    color: #9eb1a7;
    font-size: 12px;
}}

.sidebar-divider {{
    height: 1px;
    background: rgba(223,189,101,.13);
    margin: 8px 0 24px;
}}

.sidebar-section {{
    color: #dfbd65;
    font-size: 10px;
    font-weight: 800;
    letter-spacing: 2px;
    text-transform: uppercase;
    margin-bottom: 10px;
}}

div[data-testid="stMetric"] {{
    min-height: 126px;
    background: linear-gradient(145deg, rgba(10,48,34,.94), rgba(5,29,21,.94));
    border: 1px solid rgba(223,189,101,.18);
    border-radius: 18px;
    padding: 19px;
    box-shadow: 0 12px 28px rgba(0,0,0,.17);
}}

div[data-testid="stMetricLabel"] {{
    color: #a9c0b4;
    font-size: 12px;
}}

div[data-testid="stMetricValue"] {{
    color: var(--gold-light);
    font-weight: 800;
    font-family: 'Manrope', sans-serif;
}}

.stButton > button {{
    min-height: 48px;
    background: linear-gradient(110deg, #b58b36, #ead184);
    color: #10251a !important;
    border: none;
    border-radius: 13px;
    font-weight: 800;
    box-shadow: 0 8px 22px rgba(193,153,62,.16);
}}

.stButton > button:hover {{
    background: linear-gradient(110deg, #d6ae54, #ffecb1);
    transform: translateY(-1px);
}}

div[data-testid="stMetric"] {{
    background:
        linear-gradient(
            145deg,
            rgba(14,49,35,.96),
            rgba(8,30,23,.96)
        );
    border: 1px solid var(--border);
    border-radius: 16px;
    padding: 19px 18px;
    box-shadow: 0 8px 24px rgba(0,0,0,.16);
}}

div[data-testid="stMetricLabel"] {{
    color: #c4d9c8;
    font-size: 13px;
}}

div[data-testid="stMetricValue"] {{
    color: var(--gold-light);
    font-weight: 800;
    font-family: 'Manrope',sans-serif;
}}

div[data-testid="stSelectbox"]>div>div {{
    background-color: rgba(15,48,35,.95);
    color: #f8f2d9;
    border: 1px solid var(--border);
    border-radius: 10px;
}}

.stButton>button {{
    background:
        linear-gradient(
            110deg,
            #b58b36,
            #e7cc7c
        );
    color: #10251a !important;
    border: none;
    border-radius: 12px;
    font-weight: 800;
    padding: 12px 20px;
    transition: all .2s ease;
    box-shadow: 0 5px 18px rgba(193,153,62,.18);
}}

.stButton>button:hover {{
    background:
        linear-gradient(
            110deg,
            #d6ae54,
            #ffebae
        );
    transform: translateY(-2px);
    box-shadow: 0 8px 25px rgba(193,153,62,.3);
}}

.stDownloadButton>button {{
    background: rgba(22,106,73,.9);
    border: 1px solid rgba(108,201,150,.5);
    border-radius: 10px;
    color: white;
}}

div[data-testid="stDataFrame"] {{
    border: 1px solid var(--border);
    border-radius: 14px;
    overflow: hidden;
}}

hr {{
    border-color: rgba(223,189,101,.2);
}}

.stAlert {{
    border-radius: 12px;
}}

footer {{
    visibility: hidden;
}}


</style>
""", unsafe_allow_html=True)

# ============================================
# HERO SECTION
# ============================================

st.html("""
<div class="hero">
    <div class="hero-label">
        Intelligent Digital Asset Analytics
    </div>

    <div class="hero-title">
        ₿ CryptoCast
    </div>

    <div class="hero-desc">
        Explore Bitcoin price trends with deep learning.
        Compare neural network architectures, generate forecasts,
        and visualize potential market movements in one dashboard.
    </div>

    <div class="hero-badges">
        <span class="hero-badge">CNN</span>
        <span class="hero-badge">RNN</span>
        <span class="hero-badge">LSTM</span>
        <span class="hero-badge">Transformer</span>
        <span class="hero-badge">1 · 3 · 7 Day Forecasts</span>
    </div>
</div>
""")
# ============================================
# DATA LOADING
# ============================================

@st.cache_data
def load_data():
    return pd.read_csv(DATA_PATH)

try:
    df = load_data()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)

except Exception as e:
    st.error(f"Unable to load Bitcoin data: {e}")
    st.stop()

# ============================================
# MODEL LOADING
# ============================================

@st.cache_resource
def load_forecast_model(model_path):

    return load_model(
        model_path,
        custom_objects={
            "PositionalEncoding": PositionalEncoding
        },
        compile=False
    )
@st.cache_resource
def load_scaler(scaler_path):
    return joblib.load(scaler_path)


# ============================================
# FORECAST FUNCTION
# ============================================

def predict_forecast(model_type, horizon):
    model_file, scaler_file = MODEL_CONFIG[
        model_type
    ][horizon]

    model_path = os.path.join(
        MODEL_DIR, model_file
    )

    scaler_path = os.path.join(
        MODEL_DIR, scaler_file
    )

    if not os.path.exists(model_path):
        raise FileNotFoundError(
            f"Model not found: {model_path}"
        )

    if not os.path.exists(scaler_path):
        raise FileNotFoundError(
            f"Scaler not found: {scaler_path}"
        )

    model = load_forecast_model(model_path)
    scaler = load_scaler(scaler_path)

    input_data = df[FEATURES].tail(LOOKBACK).values
    scaled_data = scaler.transform(input_data)

    X = scaled_data.reshape(
        1, LOOKBACK, len(FEATURES)
    )

    prediction = model.predict(X, verbose=0)

    prediction = np.array(prediction).reshape(-1)

    # Inverse scaling using Price feature parameters.
    if hasattr(scaler, "data_min_") and hasattr(
        scaler, "data_max_"
    ):
        price_min = scaler.data_min_[0]
        price_max = scaler.data_max_[0]
        prediction = (
            prediction * (price_max - price_min)
            + price_min
        )

    return prediction


# ============================================
# HISTORICAL PRICE PERFORMANCE
# ============================================

st.markdown("""
<div class="section-label">
    Market Intelligence
</div>
<h2>Bitcoin Price Performance</h2>
""", unsafe_allow_html=True)

col1, col2, col3 = st.columns(3)

latest_price = df["Price"].iloc[-1]
previous_price = df["Price"].iloc[-2]
price_change = (
    (latest_price - previous_price)
    / previous_price
) * 100

with col1:
    st.metric(
        "Latest Bitcoin Price",
        f"${latest_price:,.2f}"
    )

with col2:
    st.metric(
        "Daily Change",
        f"{price_change:.2f}%"
    )

with col3:
    st.metric(
        "Historical Records",
        f"{len(df):,}"
    )

st.line_chart(
    df.set_index("Date")["Price"],
    color="#dfbd65"
)

# ============================================
# FORECAST SETTINGS
# ============================================

st.sidebar.markdown("""
<div class="sidebar-brand">
    <div class="sidebar-brand-row">
        <span class="sidebar-dot"></span>
        <span class="sidebar-brand-title">CryptoCast</span>
    </div>
    <div class="sidebar-subtitle">Deep Learning Forecast Control Center</div>
</div>
<div class="sidebar-divider"></div>
<div class="sidebar-section">Forecast Configuration</div>
""", unsafe_allow_html=True)

model_type = st.sidebar.selectbox(
    "Select Deep Learning Model",
    ["CNN", "RNN", "LSTM", "Transformer"]
)

horizon = st.sidebar.selectbox(
    "Forecast Horizon",
    [1, 3, 7],
    format_func=lambda x: f"{x} Day(s)"
)

# ============================================
# FORECAST SECTION
# ============================================

st.markdown(f"""
<div class="section-label">AI Prediction Engine</div>
<h2 class="forecast-heading">Forecast Your Market</h2>

<div class="forecast-meta">
    <span class="forecast-chip">Model: <strong>{model_type}</strong></span>
    <span class="forecast-chip">Horizon: <strong>{horizon} day(s)</strong></span>
    <span class="forecast-chip">Lookback: <strong>{LOOKBACK} days</strong></span>
</div>
""", unsafe_allow_html=True)

# ============================================
# GENERATE FORECAST
# ============================================

if st.button(
    "🚀 Generate Forecast",
    use_container_width=True
):
    with st.spinner(
        f"Generating {horizon}-day forecast using {model_type}..."
    ):
        try:
            forecast = predict_forecast(
                model_type,
                horizon
            )

            if len(forecast) != horizon:
                st.error(
                    f"Expected {horizon} predictions, "
                    f"but model returned {len(forecast)}."
                )
            else:
                forecast_dates = pd.date_range(
                    start=df["Date"].iloc[-1],
                    periods=horizon + 1,
                    freq="D"
                )[1:]

                forecast_df = pd.DataFrame({
                    "Date": forecast_dates,
                    "Predicted Price": forecast
                })

                st.success("Forecast generated successfully!")

                st.dataframe(
                    forecast_df.style.format({
                        "Predicted Price": "${:,.2f}"
                    }),
                    use_container_width=True
                )

                st.line_chart(
                    forecast_df.set_index("Date")[
                        "Predicted Price"
                    ],
                    color="#35c98b"
                )

                csv = forecast_df.to_csv(
                    index=False
                ).encode("utf-8")

                st.download_button(
                    "⬇️ Download Forecast CSV",
                    data=csv,
                    file_name="cryptocast_forecast.csv",
                    mime="text/csv"
                )

        except Exception as e:
            st.error(
                f"Forecast failed: {e}"
            )

# ============================================
# FOOTER
# ============================================

st.markdown("---")

st.markdown(
    """
    <div style="
        text-align:center;
        color:#dfbd65;
        padding:15px;
        font-size:13px;
    ">
        ₿ CryptoCast | Deep Learning Bitcoin Forecasting
        <br>
        CNN · RNN · LSTM · Transformer
    </div>
    """,
    unsafe_allow_html=True
)

Overwriting /content/mydrive/MyDrive/cryptocast/streamlit/app.py


In [ ]:
from pathlib import Path

app = Path("/content/mydrive/MyDrive/cryptocast/streamlit/app.py")

print("Exists:", app.exists())
print("Size:", app.stat().st_size)

code = app.read_text(encoding="utf-8")

print("Hero:", 'st.html("""' in code)
print("CryptoCast:", "₿ CryptoCast" in code)
print("Hero badges:", "hero-badges" in code)

Exists: True
Size: 17611
Hero: True
CryptoCast: True
Hero badges: True


In [ ]:
!pkill -9 -f "streamlit run" || true

^C


In [ ]:
!ps aux | grep streamlit | grep -v grep

root        1304  8.9  0.5 528036 75016 ?        Sl   13:33   0:02 python3 -m streamlit run /content/mydrive/MyDrive/cryptocast/streamlit/app.py --server.port 8501 --server.address 0.0.0.0


In [ ]:
!nohup python -m streamlit run \
"/content/mydrive/MyDrive/cryptocast/streamlit/app.py" \
--server.port 8501 \
--server.address 0.0.0.0 \
> /content/streamlit.log 2>&1 &

In [ ]:
!sleep 5
!cat /content/streamlit.log



2026-10-05 13:34:22.846 Port 8501 is not available


In [ ]:
!cloudflared tunnel --url http://localhost:8501

2026-10-05T13:35:47Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-05T13:35:47Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-05T13:35:51Z INF +--------------------------------------------------------------------------------------------+
2026-10-05T13:35:51Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-05T13:35:51Z INF |  https://achieving-away-lodging-adam.trycloudflare.com

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared

!chmod +x /usr/local/bin/cloudflared

!cloudflared --version

cloudflared version 2026.9.3 (built 2026-09-24-16:07 UTC)


In [ ]:
!ps aux | grep streamlit | grep -v grep

root        4844  1.8  0.5 527812 74436 ?        Sl   12:09   0:02 python3 -m streamlit run /content/mydrive/MyDrive/cryptocast/streamlit/app.py --server.port 8501 --server.address 0.0.0.0


In [ ]:
!nohup python -m streamlit run \
"/content/mydrive/MyDrive/cryptocast/streamlit/app.py" \
--server.port 8501 \
--server.address 0.0.0.0 \
> /content/streamlit.log 2>&1 &

In [ ]:
!ps aux | grep "streamlit run" | grep -v grep

In [ ]:
from pathlib import Path

app = Path("/content/mydrive/MyDrive/cryptocast/streamlit/app.py")
code = app.read_text(encoding="utf-8")

print("Hero badges:", "hero-badges" in code)
print("Forecast chips:", "forecast-meta" in code)
print("New sidebar:", "Deep Learning Forecast Control Center" in code)

Hero badges: True
Forecast chips: True
New sidebar: True


In [ ]:
!nohup python -m streamlit run \
"/content/mydrive/MyDrive/cryptocast/streamlit/app.py" \
--server.port 8501 \
--server.address 0.0.0.0 \
> /content/streamlit.log 2>&1 &

In [ ]:
!sleep 5
!cat /content/streamlit.log



2026-10-05 13:34:22.846 Port 8501 is not available


In [ ]:
!pkill -9 -f "streamlit" || true
!pkill -9 -f "cloudflared" || true

^C
^C


In [ ]:
from pathlib import Path

app = Path("/content/mydrive/MyDrive/cryptocast/streamlit/app.py")

print("Exists:", app.exists())
print("Size:", app.stat().st_size if app.exists() else "N/A")

if app.exists():
    code = app.read_text(encoding="utf-8")
    print("Hero badges:", "hero-badges" in code)
    print("Forecast chips:", "forecast-meta" in code)
    print("New sidebar:", "Deep Learning Forecast Control Center" in code)

Exists: True
Size: 16098
Hero badges: True
Forecast chips: True
New sidebar: True


In [ ]:
!nohup python -m streamlit run \
"/content/mydrive/MyDrive/cryptocast/streamlit/app.py" \
--server.port 8501 \
--server.address 0.0.0.0 \
> /content/streamlit.log 2>&1 &

In [ ]:
!sleep 8
!cat /content/streamlit.log



2026-10-05 13:35:03.674 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.16.237.250:8501



In [ ]:
!curl -I http://localhost:8501

HTTP/1.1 200 OK
date: Mon, 05 Oct 2026 13:35:14 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 6463
last-modified: Mon, 05 Oct 2026 13:32:36 GMT
etag: "e2cfdc4d180991be94bcf8c6b52a4a15"
cache-control: no-cache



In [ ]:
!curl -I http://127.0.0.1:8501

HTTP/1.1 200 OK
date: Mon, 05 Oct 2026 13:35:20 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 6463
last-modified: Mon, 05 Oct 2026 13:32:36 GMT
etag: "e2cfdc4d180991be94bcf8c6b52a4a15"
cache-control: no-cache



In [ ]:
!pkill -f cloudflared || true

!nohup cloudflared tunnel --url http://127.0.0.1:8501 \
> /content/cloudflared.log 2>&1 &

^C


In [ ]:
!sleep 5
!cat /content/cloudflared.log

cat: /content/cloudflared.log: No such file or directory


In [25]:
!pwd
!ls

/content
mydrive  sample_data  streamlit.log


In [26]:
!ls -la

total 24
drwxr-xr-x 1 root root 4096 Oct  5 13:33 .
drwxr-xr-x 1 root root 4096 Oct  5 13:30 ..
drwxr-xr-x 4 root root 4096 Sep 30 13:28 .config
drwx------ 5 root root 4096 Oct  5 13:33 mydrive
drwxr-xr-x 1 root root 4096 Sep 30 13:28 sample_data
-rw-r--r-- 1 root root 3514 Oct  5 13:38 streamlit.log


In [27]:
!git clone https://github.com/sanjay-prakash0706/cryptocast.git

Cloning into 'cryptocast'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 53 (delta 14), reused 53 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 9.01 MiB | 14.03 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [41]:
!find /content -name "Untitled0.ipynb" -o -name "04_Transformer_Bitcoin_Forecasting.ipynb"

/content/cryptocast/notebook/04_Transformer_Bitcoin_Forecasting.ipynb
/content/cryptocast/streamlit/Untitled0.ipynb
/content/mydrive/MyDrive/Colab Notebooks/Untitled0.ipynb
/content/mydrive/MyDrive/cryptocast/notebook/04_Transformer_Bitcoin_Forecasting.ipynb
/content/mydrive/MyDrive/cryptocast/streamlit/Untitled0.ipynb


In [42]:
%cd /content/cryptocast
!git status

/content/cryptocast
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
